# Speech Denoising — CNN v2 with Validation Loop

4-layer 1D CNN (encoder-decoder) with train/validation split.

**Architecture:** Conv1d 1→16→32→16→1 | **Loss:** MSELoss | **Optimizer:** Adam | **Epochs:** 4

---

## 1. Imports

In [1]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import librosa as lb
import matplotlib.pyplot as plt
import soundfile as sf
from pathlib import Path
from IPython.display import Audio
from tqdm import tqdm

## 2. Load Preprocessed Data

In [2]:
train_clean_chunks = np.load('data/train_clean_chunks.npy')
train_noisy_chunks = np.load('data/train_noisy_chunks.npy')

In [3]:
torch_train_noisy_chunks = torch.from_numpy(train_noisy_chunks)
torch_train_clean_chunks = torch.from_numpy(train_clean_chunks)

## 3. Dataset Split

Split trainset 80/20 into train and validation. Validation set is never seen during training — used only to monitor generalization.

In [4]:
full_dataset = torch.utils.data.TensorDataset(torch_train_noisy_chunks, torch_train_clean_chunks)

In [5]:
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [0.8, 0.2])

In [6]:
len(train_dataset),  len(val_dataset)

(22464, 5615)

In [7]:
train_dataloader = DataLoader(train_dataset, batch_size=32,)
val_dataloader = DataLoader(val_dataset, batch_size=32,)

## 4. Model Architecture

**Encoder-decoder** CNN:
1) Conv1d(1→16) — extract low-level features
2) Conv1d(16→32) — extract higher-level features
3) Conv1d(32→16) — decode back
4) Conv1d(16→1) — reconstruct clean waveform

LeakyReLU between layers (not after last). unsqueeze/squeeze handle Conv1d channel dimension.

In [10]:
class DenoisingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu = nn.LeakyReLU()
        self.layer1 = nn.Conv1d(1, 16, 3, padding=1)
        self.layer2 = nn.Conv1d(16, 32, 3, padding=1)
        self.layer3 = nn.Conv1d(32, 16, 3, padding=1)
        self.layer4 = nn.Conv1d(16, 1, 3, padding=1)
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.layer3(x)
        x = self.relu(x)
        x = self.layer4(x)
        x = x.squeeze(1)
        return x

In [15]:
model = DenoisingModel()
print(model)


DenoisingModel(
  (relu): LeakyReLU(negative_slope=0.01)
  (layer1): Conv1d(1, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (layer2): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (layer3): Conv1d(32, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (layer4): Conv1d(16, 1, kernel_size=(3,), stride=(1,), padding=(1,))
)


## 5. Loss Function & Optimizer

In [12]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001 )

## 6. Training Loop with Validation

After each epoch: switch to eval mode, compute val loss with no_grad, switch back to train mode.
If val loss rises while train loss falls — overfitting.

In [ ]:
n_epochs = 4

for epoch in tqdm(range(n_epochs)):
    epoch_loss = 0
    for batch in train_dataloader:
        noisy_batch, clean_batch = batch
        optimizer.zero_grad()
        train_pred_clean = model(noisy_batch)
        train_loss = loss_fn(train_pred_clean, clean_batch)
        train_loss.backward()
        optimizer.step()
        epoch_loss += train_loss.item()
    avg_epoch_loss = epoch_loss / len(train_dataloader)
    print(f"Epoch {epoch}, Avg Loss: {avg_epoch_loss:.4f}")
    
    model.eval()
    with torch.no_grad():
        val_epoch_loss = 0
        for batch in val_dataloader:
            noisy_val_batch, clean_val_batch = batch
            val_pred_clean = model(noisy_val_batch)
            val_loss = loss_fn(val_pred_clean, clean_val_batch)
            val_epoch_loss += val_loss.item()
        avg_val_loss = val_epoch_loss / len(val_dataloader)
        print(f"Epoch {epoch}, Avg Val Loss: {avg_val_loss:.4f}")
    model.train()    

## 7. Save Model Weights

In [ ]:
torch.save(model.state_dict(), 'denoising_cnn_4layers_4epochs.pth')

## 8. Inference

Load test file, run through trained model, compare noisy input vs predicted clean output.

In [21]:
test_noisy, _ = lb.load('data/archive/noisy_testset_wav/p232_019.wav', sr=None)
test_noisy_tensor = torch.from_numpy(test_noisy).unsqueeze(0)
# run inference
pred_clean = model(test_noisy_tensor).detach().numpy().squeeze(0)


In [22]:
# compare: noisy input vs model output
Audio(test_noisy, rate=16000)


In [23]:
Audio(pred_clean, rate=16000)
